# 04 — Negotiation & Coordination
**Agentic X | Studio 67** — Real components only. Run from the repository root.

## 1. Objective
Combine four live agents into one negotiation coordinator. Candidate options are accepted only when **all four** approve; vetoes cannot be overridden.

In [ ]:
from pathlib import Path
import sys
ROOT=Path.cwd()
if not (ROOT/'core').exists():ROOT=ROOT.parent
sys.path.insert(0,str(ROOT))
from agents.creative import CreativeAgent
from agents.finance import FinanceAgent
from agents.safety import SafetyAgent
from agents.operations import OperationsAgent
from core.environment import get_case
from core.negotiation import NegotiationEngine

## 2. Instantiate real agents
The LLM-based agents require `OPENAI_API_KEY`. There is no mock mode.

In [ ]:
creative=CreativeAgent(); finance=FinanceAgent(); safety=SafetyAgent(); operations=OperationsAgent()
engine=NegotiationEngine(creative,finance,safety,operations,max_rounds=3)
print([agent.name for agent in engine.agents])

## 3. Execute actual negotiations
The engine generates messages **after** assessing each option, not from predefined dialogue.

In [ ]:
case, options=get_case('SCR-001')
result=engine.run(case,options)
print('Status:',result.status,'Selected:',result.selected_option_id,'Rounds:',result.rounds)
print('Messages:',len(result.messages),'Evaluations:',len(result.evaluations))

## 4. Inspect generated ACL dialogue
Explain causal connections between each evaluation and its acceptance/refusal.

In [ ]:
for message in result.messages:
    print(f'R{message.round}: {message.sender} -> {message.receiver} [{message.performative}] {message.content}')

## 5. Enforce safety/finance vetoes
If an option is chosen, every real evaluation for that option must approve it.

In [ ]:
if result.selected_option_id:
    selected=[x for x in result.evaluations if x.option_id==result.selected_option_id]
    assert len(selected)==4 and all(x.approved for x in selected)
else:
    assert result.status=='ESCALATED'
assert result.is_mock is False
print('Negotiation invariants passed')

## 6. Known boundary
Current negotiation filters and ranks options supplied by the dataset. It does **not yet** modify options or generate counteroffers; it escalates when all candidates are refused. Extend to real multi-round counter-proposals before claiming adaptive negotiation.

In [ ]:
print('Current engine: four independently evaluated agents, ACL log, accept/refuse, and escalation.')